# Geometric Hallucination Detection with SGI on HaluEval

An independent reproduction attempt of the context-based Semantic Grounding Index (SGI) discussed in [Javier Marín, *A Geometric Taxonomy of Hallucinations in LLMs*](https://arxiv.org/html/2602.13224v1). This notebook focuses on HaluEval only. It does not implement the paper's context-free Directional Grounding Index.

For normalized embeddings, angular distance is `theta(a, b) = arccos(clip(a · b, -1, 1))`, and `SGI = theta(response, question) / theta(response, context)`. Higher scores suggest stronger context grounding; this is a geometric signal, not a factuality guarantee.

## Recorded progress before cleanup

The original saved outputs reported 64,507 normalized response rows: 20,000 each for QA, dialogue and summarization, plus 4,507 general-query rows. Filtering yielded 60,000 complete triplets. The embedding cell reported a cache hit for `all-mpnet-base-v2`. The external Drive cache is not included in this repository and has not been independently verified here. No SGI values, AUROC or effect sizes were saved in the original notebook.

The scoring and evaluation sections below were added during cleanup. A subsequent Colab run on November 4, 2026 produced results now recorded in README.md and results/HaluEval/. This notebook now includes saved outputs from the subsequent Colab evaluation and threshold sweep. All execution outputs were cleared to avoid presenting historical outputs as runs of the revised code.


## 1. Environment

Run in Google Colab with a GPU and Google Drive access. CPU execution is supported but slower. Downloads require internet access. Dependencies are installed without version pins; record the generated environment file when comparing runs.


In [1]:
%pip -q install transformers torch numpy pandas pyarrow scikit-learn tqdm

import gc
import hashlib
import importlib.metadata
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from google.colab import drive
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "HaluEval"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "by_dataset" / "HaluEval"
RESULTS_DIR = PROJECT_ROOT / "results" / "HaluEval"
for path in [RAW_DIR.parent, PROCESSED_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
versions = {name: importlib.metadata.version(name) for name in
            ["transformers", "torch", "numpy", "pandas", "pyarrow", "scikit-learn"]}
(RESULTS_DIR / "environment.json").write_text(json.dumps(versions, indent=2))
print("Device:", DEVICE)


Mounted at /content/drive
Device: cuda


## 2. Download HaluEval

Clone the official repository once and record its revision. Existing checkouts are reused without pulling updates.


In [2]:
if not (RAW_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/RUCAIBox/HaluEval.git", str(RAW_DIR)], check=True)
DATA_REVISION = subprocess.check_output(
    ["git", "-C", str(RAW_DIR), "rev-parse", "HEAD"], text=True).strip()
print("HaluEval revision:", DATA_REVISION)


HaluEval revision: b7253db3cdaa0ab2c382f92b26b390109174f77e


## 3. Normalize response pairs

Each task-specific source item becomes two rows: a reference response (`label=0`) and a hallucinated response (`label=1`). Dialogue history serves as the question. Summarization uses the fixed instruction `Summarize the document faithfully.` as a synthetic question, so its geometry is not directly equivalent to QA. General queries have no context and are retained only in the processed data, not scored with SGI.


In [3]:
def normalize_text(value):
    return " ".join(str(value).split()) if value is not None else ""


def load_records(path):
    text = path.read_text(encoding="utf-8").strip()
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, list) else [obj]
    except json.JSONDecodeError:
        return [json.loads(line) for line in text.splitlines() if line.strip()]


rows = []
settings = {
    "qa": ("question", "knowledge", "right_answer", "hallucinated_answer"),
    "dialogue": ("dialogue_history", "knowledge", "right_response", "hallucinated_response"),
    "summarization": (None, "document", "right_summary", "hallucinated_summary"),
}
for subset, (q_key, c_key, right_key, hallu_key) in settings.items():
    source = RAW_DIR / "data" / f"{subset}_data.json"
    for i, item in enumerate(load_records(source)):
        q = normalize_text(item.get(q_key)) if q_key else "Summarize the document faithfully."
        c = normalize_text(item.get(c_key))
        pair_id = f"{subset}_{i}"
        for label, key, suffix in [(0, right_key, "right"), (1, hallu_key, "hallu")]:
            r = normalize_text(item.get(key))
            if q and r:
                rows.append(dict(dataset="HaluEval", subset=subset, pair_id=pair_id,
                                 sample_id=f"{pair_id}_{suffix}", question=q, context=c,
                                 response=r, label=label, source_file=str(source)))
source = RAW_DIR / "data" / "general_data.json"
for i, item in enumerate(load_records(source)):
    q, r = normalize_text(item.get("user_query")), normalize_text(item.get("chatgpt_response"))
    value = normalize_text(item.get("hallucination_label", item.get("hallucination"))).lower()
    if q and r:
        rows.append(dict(dataset="HaluEval", subset="general", pair_id=f"general_{i}",
                         sample_id=f"general_{i}", question=q, context="", response=r,
                         label={"yes": 1, "no": 0}.get(value, np.nan), source_file=str(source)))
df = pd.DataFrame(rows)
df.to_parquet(PROCESSED_DIR / "native.parquet", index=False)
df.to_csv(PROCESSED_DIR / "native.csv", index=False)
mask = df[["question", "context", "response"]].ne("").all(axis=1) & df.label.isin([0, 1])
df_ready = df.loc[mask].reset_index(drop=True)
assert len(df_ready) > 0, "No complete labeled triplets found."
display(df.groupby("subset").size().rename("processed_rows").to_frame())
display(df_ready.groupby(["subset", "label"]).size().rename("scored_rows").to_frame())


,processed_rows
subset,
dialogue,20000
general,4507
qa,20000
summarization,20000


scored_rows
subset        label             
dialogue      0            10000
              1            10000
qa            0            10000
              1            10000
summarization 0            10000
              1            10000

## 4. Embed and cache complete triplets

Use the original default encoder and preprocessing: `all-mpnet-base-v2`, attention-mask-aware mean pooling, L2 normalization, a 384-token limit and batches of 32. Long contexts are truncated. Cache reuse requires matching input content and configuration; legacy caches without this provenance are recomputed.


In [4]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
MAX_LENGTH = 384
BATCH_SIZE = 32
CACHE_VERSION = "manual_meanpool_l2_v2"
model_slug = MODEL_NAME.replace("/", "_")
CACHE_DIR = PROJECT_ROOT / "embeddings" / CACHE_VERSION / model_slug / "HaluEval" / "complete_triplets"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fields = ["question", "context", "response"]
fingerprint = hashlib.sha256(df_ready.to_json(orient="records").encode()).hexdigest()
config = dict(model_name=MODEL_NAME, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
              pooling="manual_mean_pooling", normalization="l2", rows=len(df_ready),
              input_sha256=fingerprint, data_revision=DATA_REVISION, environment=versions)
config_path = CACHE_DIR / "config.json"
cache_valid = (config_path.exists() and json.loads(config_path.read_text()) == config
               and all((CACHE_DIR / f"{field}.npy").exists() for field in fields)
               and (CACHE_DIR / "metadata.parquet").exists())


def mean_pooling(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


if not cache_valid:
    # Invalidate the completion marker before overwriting any artifacts.
    config_path.unlink(missing_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    for field in fields:
        parts = []
        texts = df_ready[field].tolist()
        for start in tqdm(range(0, len(texts), BATCH_SIZE), desc=field):
            encoded = tokenizer(texts[start:start+BATCH_SIZE], padding=True,
                                truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
            with torch.inference_mode():
                output = model(**encoded)
                pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
                parts.append(F.normalize(pooled, p=2, dim=1).cpu().numpy().astype(np.float32))
        np.save(CACHE_DIR / f"{field}.npy", np.vstack(parts))
    df_ready.to_parquet(CACHE_DIR / "metadata.parquet", index=False)
    config_path.write_text(json.dumps(config, indent=2))
    del model, tokenizer, parts, encoded, output, pooled
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Validated cache hit:", CACHE_DIR)


Validated cache hit: /content/drive/MyDrive/Hallucination_Project/embeddings/manual_meanpool_l2_v2/sentence-transformers_all-mpnet-base-v2/HaluEval/complete_triplets


## 5. Compute SGI and descriptive evaluation

Evaluation stage. Summary results from the first Colab run are recorded in README.md and results/HaluEval/. Lower SGI is treated as more hallucination-like, so AUROC uses `-SGI` with hallucination as the positive class. Report each subset separately; the pooled result is descriptive. Cohen's d uses `(mean_grounded - mean_hallucinated) / pooled_standard_deviation`.

No classifier or threshold is fitted. These are full-dataset descriptive measurements, not held-out classifier results. A bootstrap or future train/test split must group both responses by `pair_id`. Near-zero response–context angles are excluded because the ratio is unstable.


In [5]:
from sklearn.metrics import roc_auc_score


def angular_distance(a, b):
    return np.arccos(np.clip(np.sum(a.astype(np.float64) * b.astype(np.float64), axis=1), -1.0, 1.0))


q, c, r = [np.load(CACHE_DIR / f"{field}.npy") for field in fields]
assert q.shape == c.shape == r.shape and q.shape[0] == len(df_ready)
for emb in [q, c, r]:
    assert np.isfinite(emb).all() and np.allclose(np.linalg.norm(emb, axis=1), 1, atol=1e-5)
scored = df_ready.copy()
scored["theta_rq"] = angular_distance(r, q)
scored["theta_rc"] = angular_distance(r, c)
scored["theta_qc"] = angular_distance(q, c)
ANGLE_EPS = 1e-6
valid = scored.theta_rc > ANGLE_EPS
scored["sgi"] = np.nan
scored.loc[valid, "sgi"] = scored.loc[valid, "theta_rq"] / scored.loc[valid, "theta_rc"]
scored.to_parquet(RESULTS_DIR / "sgi_scores.parquet", index=False)


def summarize(name, frame):
    usable = frame.dropna(subset=["sgi"])
    grounded = usable.loc[usable.label == 0, "sgi"].to_numpy()
    hallucinated = usable.loc[usable.label == 1, "sgi"].to_numpy()
    d, auc = np.nan, np.nan
    if len(grounded) > 1 and len(hallucinated) > 1:
        pooled_var = ((len(grounded)-1)*grounded.var(ddof=1)
                      + (len(hallucinated)-1)*hallucinated.var(ddof=1)) / (len(usable)-2)
        if pooled_var > 0:
            d = (grounded.mean()-hallucinated.mean()) / np.sqrt(pooled_var)
    if usable.label.nunique() == 2:
        auc = roc_auc_score(usable.label, -usable.sgi)
    return dict(subset=name, rows=len(frame), excluded_rows=len(frame)-len(usable),
                grounded_rows=len(grounded), hallucinated_rows=len(hallucinated),
                mean_sgi_grounded=grounded.mean() if len(grounded) else np.nan,
                mean_sgi_hallucinated=hallucinated.mean() if len(hallucinated) else np.nan,
                cohens_d=d, auroc=auc)


metrics = pd.DataFrame([summarize(name, frame) for name, frame in scored.groupby("subset")]
                       + [summarize("all", scored)])
metrics.to_csv(RESULTS_DIR / "sgi_metrics.csv", index=False)
(RESULTS_DIR / "run_config.json").write_text(json.dumps(dict(config, angle_epsilon=ANGLE_EPS,
    positive_label=1, auroc_score="-sgi", seed=SEED), indent=2))
display(metrics)


,subset,rows,excluded_rows,grounded_rows,hallucinated_rows,mean_sgi_grounded,mean_sgi_hallucinated,cohens_d,auroc
0,dialogue,20000,0,10000,10000,2.142929,1.057075,0.018896,0.474465
1,qa,20000,0,10000,10000,1.066184,0.885615,0.805753,0.739861
2,summarization,20000,0,10000,10000,2.500901,2.683187,-0.313486,0.408410
3,all,60000,0,30000,30000,1.903338,1.541959,0.010889,0.539721


## 6. Compare thresholds and class-mean midpoints

After mounting Drive, this section reads saved `sgi_scores.parquet`. No model or GPU is needed. In a fresh runtime install numpy, pandas and pyarrow first. Set PROJECT_ROOT in the cell to match your folder.

The fixed grid is **0.70 to 1.40, step 0.05**, with reference threshold 1.00. The default prediction remains hallucination when SGI < threshold; equality is grounded.

The section also computes both class means for each task and their midpoint. Each midpoint is evaluated even when it lies outside the grid. A separately labeled diagnostic checks the direction implied by the class means, so the reversed summarization means do not silently change the default rule. These are same-data exploratory estimates, not optimal or validated thresholds.

Exports go to `results/HaluEval/thresholds_0.70_1.40/`, preserving the completed sweep in the parent folder. The current cell has no saved results for this revision. Earlier cells retain the completed SGI outputs.


In [ ]:
# Run after mounting Drive; no model or GPU is needed for saved-score analysis.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Use Drive in Colab, or the repository directory in a local VS Code kernel.
import os
cloud_root = Path("/content/drive/MyDrive/Hallucination_Project")
PROJECT_ROOT = Path(os.environ.get("SGI_PROJECT_ROOT", str(cloud_root if cloud_root.exists() else Path.cwd())))
RESULTS_DIR = PROJECT_ROOT / "results" / "HaluEval"
SCORES_PATH = RESULTS_DIR / "sgi_scores.parquet"
THRESHOLD_START = 0.70
THRESHOLD_STOP = 1.40
THRESHOLD_STEP = 0.05
REFERENCE_THRESHOLD = 1.00
TASKS = ["qa", "dialogue", "summarization", "all"]

start_units = int(round(THRESHOLD_START * 100))
stop_units = int(round(THRESHOLD_STOP * 100))
step_units = int(round(THRESHOLD_STEP * 100))
if step_units <= 0 or stop_units < start_units:
    raise ValueError("Use a positive step and stop >= start.")
if (stop_units - start_units) % step_units:
    raise ValueError("The step must reach the upper endpoint exactly.")
THRESHOLDS = np.arange(start_units, stop_units + 1, step_units) / 100
if not np.any(np.isclose(THRESHOLDS, REFERENCE_THRESHOLD)):
    raise ValueError("The grid must contain the reference threshold 1.00.")
# Separate outputs preserve the completed 1.00-2.00 sweep.
OUTPUT_DIR = RESULTS_DIR / f"thresholds_{THRESHOLD_START:.2f}_{THRESHOLD_STOP:.2f}"


def threshold_metrics(frame, thresholds, prediction_direction="below", tasks=TASKS):
    if prediction_direction not in {"below", "above"}:
        raise ValueError("prediction_direction must be below or above")
    required = {"subset", "sgi", "label"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing score columns: {sorted(missing)}")
    scores = pd.to_numeric(frame["sgi"], errors="coerce")
    labels = pd.to_numeric(frame["label"], errors="coerce")
    valid = np.isfinite(scores) & labels.isin([0, 1])
    work = frame.loc[valid, ["subset"]].copy()
    work["sgi"] = scores.loc[valid]
    work["label"] = labels.loc[valid].astype(int)
    result = []
    for task in tasks:
        original = frame if task == "all" else frame.loc[frame["subset"] == task]
        group = work if task == "all" else work.loc[work["subset"] == task]
        truth = group["label"].to_numpy() == 1
        values = group["sgi"].to_numpy()
        for threshold in thresholds:
            predicted = values < float(threshold) if prediction_direction == "below" else values > float(threshold)
            tp = int(np.sum(predicted & truth))
            fp = int(np.sum(predicted & ~truth))
            tn = int(np.sum(~predicted & ~truth))
            fn = int(np.sum(~predicted & truth))
            n = len(group)
            result.append(dict(
                subset=task, threshold=float(threshold), rows=len(original),
                evaluated_rows=n, excluded_rows=len(original)-n,
                tp=tp, fp=fp, tn=tn, fn=fn,
                accuracy=(tp+tn)/n if n else np.nan,
                precision=tp/(tp+fp) if tp+fp else np.nan,
                recall=tp/(tp+fn) if tp+fn else np.nan,
                f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan,
                predicted_hallucination_rate=(tp+fp)/n if n else np.nan,
            ))
    return pd.DataFrame(result)


def accuracy_markdown(table):
    lines = ["| SGI threshold | QA accuracy | Dialogue accuracy | Summarization accuracy | All accuracy |",
             "| ---: | ---: | ---: | ---: | ---: |"]
    pivot = table.pivot(index="threshold", columns="subset", values="accuracy")
    for threshold, row in pivot.sort_index().iterrows():
        cells = [f"{100*row[task]:.2f}%" if pd.notna(row[task]) else "NaN" for task in TASKS]
        lines.append(f"| {threshold:.2f} | " + " | ".join(cells) + " |")
    return "\n".join(lines)

def estimate_midpoints(frame):
    scores = pd.to_numeric(frame["sgi"], errors="coerce")
    labels = pd.to_numeric(frame["label"], errors="coerce")
    clean = frame.loc[np.isfinite(scores) & labels.isin([0, 1]), ["subset"]].copy()
    clean["sgi"] = scores.loc[clean.index]
    clean["label"] = labels.loc[clean.index].astype(int)
    estimates = []
    for task in TASKS:
        group = clean if task == "all" else clean.loc[clean["subset"] == task]
        grounded = group.loc[group.label == 0, "sgi"]
        hallucinated = group.loc[group.label == 1, "sgi"]
        gmean, hmean = grounded.mean(), hallucinated.mean()
        midpoint = (gmean + hmean) / 2
        # This direction is a separate diagnostic, not a change to the default rule.
        direction = "above" if hmean > gmean else "below"
        estimates.append(dict(
            subset=task, grounded_rows=len(grounded), hallucinated_rows=len(hallucinated),
            mean_sgi_grounded=gmean, mean_sgi_hallucinated=hmean,
            midpoint_threshold=midpoint, class_mean_direction=direction,
            outside_grid=bool(midpoint < THRESHOLDS.min() or midpoint > THRESHOLDS.max())
                         if np.isfinite(midpoint) else None,
        ))
    return pd.DataFrame(estimates)


def evaluate_midpoints(frame, estimates):
    rows = []
    for estimate in estimates.to_dict("records"):
        task, threshold = estimate["subset"], estimate["midpoint_threshold"]
        for policy, direction in [("fixed_below", "below"),
                                  ("class_mean_direction", estimate["class_mean_direction"])]:
            if np.isfinite(threshold):
                row = threshold_metrics(frame, [threshold], direction, tasks=[task]).iloc[0].to_dict()
            else:
                # Missing classes cannot produce a midpoint or a decision evaluation.
                row = dict(subset=task, threshold=np.nan)
            row.update(policy=policy, prediction_direction=direction, **{
                k: v for k, v in estimate.items() if k != "subset"
            })
            rows.append(row)
    return pd.DataFrame(rows)


def midpoint_markdown(estimates, evaluated):
    lines = ["| Task | Mean SGI: grounded | Mean SGI: hallucinated | Midpoint | Accuracy: SGI < midpoint | Mean-based rule | Accuracy: mean-based rule |",
             "| --- | ---: | ---: | ---: | ---: | --- | ---: |"]
    for e in estimates.to_dict("records"):
        task = e["subset"]
        fixed = evaluated.loc[(evaluated.subset == task) & (evaluated.policy == "fixed_below")]
        oriented = evaluated.loc[(evaluated.subset == task) & (evaluated.policy == "class_mean_direction")]
        def percent(frame):
            value = frame.iloc[0].get("accuracy", np.nan)
            return f"{100*value:.2f}%" if pd.notna(value) else "NaN"
        rule = "SGI > midpoint" if e["class_mean_direction"] == "above" else "SGI < midpoint"
        lines.append(f"| {task} | {e['mean_sgi_grounded']:.6f} | {e['mean_sgi_hallucinated']:.6f} | "
                     f"{e['midpoint_threshold']:.6f} | {percent(fixed)} | {rule} | {percent(oriented)} |")
    return "\n".join(lines)


if not SCORES_PATH.exists():
    raise FileNotFoundError(f"No saved scores at {SCORES_PATH}. Mount Drive and run SGI scoring first.")
score_frame = pd.read_parquet(SCORES_PATH).reset_index(drop=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
threshold_table = threshold_metrics(score_frame, THRESHOLDS)
threshold_table.to_csv(OUTPUT_DIR / "threshold_metrics.csv", index=False)
comparison = threshold_table.pivot(index="threshold", columns="subset", values="accuracy")[TASKS]
comparison.to_csv(OUTPUT_DIR / "threshold_accuracy.csv", float_format="%.8f")
accuracy_change = comparison.sub(comparison.loc[REFERENCE_THRESHOLD]).mul(100)
accuracy_change.to_csv(OUTPUT_DIR / "threshold_accuracy_change_pp.csv", float_format="%.8f")
midpoint_estimates = estimate_midpoints(score_frame)
midpoint_estimates.to_csv(OUTPUT_DIR / "midpoint_estimates.csv", index=False)
midpoint_results = evaluate_midpoints(score_frame, midpoint_estimates)
midpoint_results.to_csv(OUTPUT_DIR / "midpoint_metrics.csv", index=False)

sha256 = hashlib.sha256()
with SCORES_PATH.open("rb") as score_file:
    for block in iter(lambda: score_file.read(1024*1024), b""):
        sha256.update(block)
sweep_config = dict(
    created_utc=datetime.now(timezone.utc).isoformat(),
    scores_file=SCORES_PATH.name, scores_sha256=sha256.hexdigest(),
    threshold_start=THRESHOLD_START, threshold_stop=THRESHOLD_STOP,
    threshold_step=THRESHOLD_STEP, thresholds=THRESHOLDS.tolist(),
    reference_threshold=REFERENCE_THRESHOLD,
    prediction_rule="hallucination if sgi < threshold; equality is grounded",
    midpoint_formula="(mean_sgi_grounded + mean_sgi_hallucinated) / 2",
    midpoint_policies=["fixed_below", "class_mean_direction"],
    midpoint_clipped_to_grid=False,
    positive_label=1, evaluation="descriptive: thresholds estimated and evaluated on the same scores",
    nonfinite_scores="excluded", undefined_ratios="NaN",
)
(OUTPUT_DIR / "threshold_sweep_config.json").write_text(json.dumps(sweep_config, indent=2))

report = f"""# Experiment analysis

## Scope

SGI is evaluated on HaluEval QA, dialogue and summarization. This is not a full replication of the paper. Dialogue uses the conversation history as q; summarization uses a fixed instruction added by this project.

## Approach 1: fixed threshold grid

Predict hallucination when SGI < threshold; equality is grounded. The current grid is {THRESHOLD_START:.2f} to {THRESHOLD_STOP:.2f}, inclusive, in steps of {THRESHOLD_STEP:.2f}. Accuracy changes are measured against threshold {REFERENCE_THRESHOLD:.2f} in percentage points. These results describe the saved labeled scores, not an independent test set.

{accuracy_markdown(threshold_table)}

## Approach 2: midpoint of class means

For each task, threshold = (mean grounded SGI + mean hallucinated SGI) / 2. The pooled row is an additional summary, not a replacement for task-specific thresholds. Midpoints are evaluated at full precision and are not clipped to the grid. Means, thresholds and metrics use only finite SGI values with binary labels.

{midpoint_markdown(midpoint_estimates, midpoint_results)}

The fixed-below column preserves the original SGI rule. The mean-based diagnostic flags scores above the midpoint when the hallucinated mean exceeds the grounded mean. Equality is grounded in both policies. This second policy changes the decision direction, not the score or the baseline AUROC. When the direction is below, the policies are identical.

The midpoint is a heuristic, not an optimal threshold. It ignores dispersion, extreme ratios, class priors and error costs. The means, threshold and diagnostic direction were estimated on the same data used for evaluation, so these are exploratory results. Held-out performance remains unknown. Do not report the highest grid accuracy as validated test accuracy.

## Interpretation and next checks

The original baseline AUROC was 0.7399 for QA, 0.4745 for dialogue and 0.4084 for summarization using -SGI. The new cutoffs do not change that ranking. If the summarization midpoint lies outside the grid, it is still evaluated separately. The test asks whether a different cutoff helps, and whether the reversed mean order warrants a separate direction check. It does not establish why the ranking is reversed.

- [x] Implement and execute the 0.70-1.40 grid on saved scores.
- [x] Compute class means, midpoint thresholds and decision metrics.
- [ ] Review whether the midpoint improves accuracy, recall and false alarm rate against threshold 1.00.
- [ ] Inspect dispersion and extreme ratios, especially for dialogue.
- [ ] Select a threshold and direction on development pairs and evaluate once on separate test pairs.
- [ ] Add bootstrap intervals grouped by pair_id.

## Run evidence

This report was generated from saved per-response scores. threshold_sweep_config.json records their SHA-256 and the evaluation settings. threshold_metrics.csv contains the grid metrics; midpoint_estimates.csv and midpoint_metrics.csv contain the heuristic estimates and decisions. Accuracy values in CSV are fractions; this report shows percentages. Zero-denominator ratios are NaN. Invalid rows are excluded and counted. The completed 1.00-2.00 exports remain in the parent results folder.
"""
(OUTPUT_DIR / "ANALYSIS.md").write_text(report)
display(comparison)
display(accuracy_change)
display(midpoint_estimates)
display(midpoint_results)
display(Markdown(accuracy_markdown(threshold_table)))
display(Markdown(midpoint_markdown(midpoint_estimates, midpoint_results)))
print("Saved grid, midpoint metrics and ANALYSIS.md to:", OUTPUT_DIR)


## Interpretation

Compare the QA row with the paper's SGI results; dialogue and summarization are additional task-specific measurements. An AUROC near 0.5 indicates weak ranking ability. SGI > 1 only means the response is angularly closer to context than to question; it is not a calibrated truth threshold. Report preprocessing, truncation, exclusions and model choice alongside any metrics. See README.md for the recorded project status and research reference.
